In [22]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import os
import warnings

warnings.filterwarnings("ignore")

# === Load Dataset ===
df = pd.read_csv("Stock_Preprocessed.csv")

# === Clean and Prepare ===
df.drop_duplicates(inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(['Stocks', 'Date'], inplace=True)

# === Ensure Output Directory Exists ===
os.makedirs("SARIMA_Results", exist_ok=True)

# === Loop through each stock ===
forecast_horizon = 30
all_forecasts = []

for stock in df['Stocks'].unique():
    stock_df = df[df['Stocks'] == stock].copy()
    stock_df.set_index('Date', inplace=True)

    ts = stock_df['Close']

    # Log transform to stabilize variance
    ts_log = np.log(ts)

    # Make stationary if needed (ADF test)
    adf_result = adfuller(ts_log)
    if adf_result[1] > 0.05:
        ts_log_diff = ts_log.diff().dropna()
    else:
        ts_log_diff = ts_log

    # === Fit SARIMA ===
    try:
        model = SARIMAX(ts_log_diff, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
        model_fit = model.fit(disp=False)

        forecast = model_fit.get_forecast(steps=forecast_horizon)
        forecast_log = forecast.predicted_mean
        conf_int = forecast.conf_int()

        # Reverse differencing if differenced
        if adf_result[1] > 0.05:
            last_log = ts_log.iloc[-1]
            forecast_log = forecast_log.cumsum() + last_log

        forecast_actual = np.exp(forecast_log)
        conf_int_exp = np.exp(conf_int)

        # Save forecast
        forecast_df = pd.DataFrame({
            'Stock': stock,
            'Forecast_Date': pd.date_range(ts.index[-1] + pd.Timedelta(days=1), periods=forecast_horizon),
            'Predicted_Close': forecast_actual.values
        })

        forecast_df.to_csv(f"SARIMA_Results/{stock}_forecast.csv", index=False)
        all_forecasts.append(forecast_df)

    except Exception as e:
        print(f"Error for {stock}: {e}")

# Save all forecasts together (optional)
pd.concat(all_forecasts).to_csv("SARIMA_Results/All_Forecasts.csv", index=False)
print("SARIMA forecasting complete.")


PermissionError: [Errno 13] Permission denied: 'Stock_Preprocessed.csv'